# Master Brain Pipeline Playbook (Clean Run)

This notebook executes the governance pipeline from a clean baseline, then performs API-backed **Business Brain** and **Math Brain** checks.

In [11]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import pandas as pd
import requests

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'projects').exists() else cwd.parent
PIPELINE = ROOT / 'projects' / 'master_brain.py'
OUT_DIR = ROOT / 'master_brain'
BASE_URL = 'http://127.0.0.1:8787'

assert PIPELINE.exists(), f'Missing pipeline script: {PIPELINE}'
print('Notebook cwd:', cwd)
print('Repo root:', ROOT)

Notebook cwd: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\notebooks
Repo root: C:\Users\dunnm\source\repos\finance-scrapers-portfolio


In [12]:
# hard reset of prior pipeline outputs for a true clean start
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Clean output directory ready:', OUT_DIR)

cmd = [
    sys.executable,
    str(PIPELINE),
    '--in-root', str(ROOT),
    '--out-dir', str(OUT_DIR),
    '--apply-cleanup',
    '--cleanup-mode', 'move',
    '--gate-stress-bps', '20',
    '--stress-bps-grid', '10,20,35',
]

print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True, cwd=str(ROOT))
print('return_code:', res.returncode)
print(res.stdout[-3000:])
if res.returncode != 0:
    print('stderr_tail:')
    print(res.stderr[-3000:])
    raise RuntimeError('master_brain pipeline failed')

Clean output directory ready: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain
Running: c:\Users\dunnm\anaconda3\python.exe C:\Users\dunnm\source\repos\finance-scrapers-portfolio\projects\master_brain.py --in-root C:\Users\dunnm\source\repos\finance-scrapers-portfolio --out-dir C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain --apply-cleanup --cleanup-mode move --gate-stress-bps 20 --stress-bps-grid 10,20,35
return_code: 0
Wrote: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain\finance_principles_report.csv
Wrote: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain\ml_principles_report.csv
Wrote: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain\promotion_gate_report.csv
Wrote: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain\monitoring_metrics_report.csv
Wrote: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain\folder_cleanup_plan.csv
Wrote: C:\Users\dunnm\sou

In [14]:
summary = json.loads((OUT_DIR / 'master_brain_summary.json').read_text(encoding='utf-8'))
quality = pd.read_csv(OUT_DIR / 'quality_metrics_report.csv')
monitoring = pd.read_csv(OUT_DIR / 'monitoring_metrics_report.csv')
gate = pd.read_csv(OUT_DIR / 'promotion_gate_report.csv')
cleanup_exec = pd.read_csv(OUT_DIR / 'folder_cleanup_execution_report.csv')

print('counts:', summary.get('counts', {}))
display(quality)
display(monitoring.head(20))
display(gate[['run','decision','promotion_score']].head(20))
display(cleanup_exec[['category','status']].value_counts().rename('count').reset_index().head(20))

counts: {'backtests_scored': 296, 'ml_runs_scored': 6, 'promoted_runs': 0, 'cleanup_moves_planned': 0, 'cleanup_actions_executed': 0, 'cleanup_errors': 0, 'cleanup_remaining': 0}


,metric,value
0,backtests_scored,296.000000
1,ml_runs_scored,6.000000
2,gated_runs,296.000000
3,reliability_non_null_rate,1.000000
4,reliability_mean,0.270174
5,principles_clean_rate,0.402027
6,ml_quality_non_null_rate,1.000000
7,ml_quality_mean,0.472694
8,cv_present_rate,0.833333
9,promotion_score_non_null_rate,1.000000


,metric,value
0,runs_total,296.000000
1,runs_promote,0.000000
2,runs_candidate,1.000000
3,runs_reject,295.000000
4,promotion_score_median,0.125125
5,reliability_score_median,0.227500
6,ml_quality_score_median,0.000000
7,sharpe_median,0.139892
8,max_drawdown_median,-0.062730
9,profit_factor_median,1.113339


,run,decision,promotion_score
0,bt_crypto_5y_SOL-USD_rf_wv_dwt_db4,candidate,0.475828
1,bt_eurusd_sweep_EURUSD=X_hgb_baseline,reject,0.471614
2,bt_eurusd_sweep_EURUSD=X_mlp_baseline,reject,0.471614
3,bt_eurusd_sweep_EURUSD=X_hgb_stops,reject,0.466018
4,bt_eurusd_sweep_EURUSD=X_mlp_stops,reject,0.466018
5,bt_crypto_5y_BTC-USD_rf_proba_hmm_wavelet,reject,0.394404
6,bt_eurusd_sweep_EURUSD=X_rf_stops,reject,0.379244
7,bt_crypto_5y_ETH-USD_hgb_wv_dwt_db6,reject,0.372346
8,bt_crypto_5y_SOL-USD_hgb_wv_dwt_db4,reject,0.370906
9,bt_crypto_5y_SOL-USD_mlp_wv_dwt_db4,reject,0.363291


,category,status,count


In [17]:
# API-backed Business Brain + Math Brain logic checks
API_DIR = Path(r"C:\Users\dunnm\Downloads\Master-Brain-API")
KEY_NAMES = ['MASTER_BRAIN_API_KEY', 'BRIDGE_API_KEY', 'X_API_KEY', 'API_KEY']

def _load_key_from_env_file(path: Path, key_names: list[str]) -> tuple[str | None, str | None]:
    if not path.exists():
        return None, None
    try:
        lines = path.read_text(encoding='utf-8').splitlines()
    except Exception:
        return None, None

    kv = {}
    for line in lines:
        s = line.strip()
        if not s or s.startswith('#') or '=' not in s:
            continue
        k, v = s.split('=', 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        kv[k] = v

    for k in key_names:
        val = kv.get(k)
        if val and val.strip() and not any(tok in val for tok in ['YOUR_', 'PLACEHOLDER', 'CHANGE_ME', '<']):
            return val, f'{path}::{k}'
    return None, None

API_KEY = None
API_KEY_SOURCE = None

# 1) process env
for k in KEY_NAMES:
    v = os.getenv(k)
    if v and v.strip() and not any(tok in v for tok in ['YOUR_', 'PLACEHOLDER', 'CHANGE_ME', '<']):
        API_KEY = v
        API_KEY_SOURCE = f'process_env::{k}'
        break

# 2) .env fallback search
if not API_KEY:
    for env_path in [ROOT / '.env', API_DIR / '.env', API_DIR / '.env.local', API_DIR / '.env.development']:
        v, src = _load_key_from_env_file(env_path, KEY_NAMES)
        if v:
            API_KEY = v
            API_KEY_SOURCE = src
            break

if API_KEY:
    masked = API_KEY[:4] + '...' + API_KEY[-2:] if len(API_KEY) >= 8 else '***'
    print('API key source:', API_KEY_SOURCE)
    print('API key (masked):', masked, '| length:', len(API_KEY))
else:
    print('No usable API key found. Populate BRIDGE_API_KEY in .env or process environment.')

headers = {'x-api-key': API_KEY} if API_KEY else {}

health = requests.get(f'{BASE_URL}/health', timeout=10)
print('health_status:', health.status_code)
print('health_json:', health.json())

snapshot = {
    'counts': summary.get('counts', {}),
    'quality_metrics': dict(zip(quality['metric'], quality['value'])),
    'monitoring_metrics': dict(zip(monitoring['metric'], monitoring['value'])),
}

queries = [
    {
        'check': 'business_brain_finance_principles',
        'question': 'Use Business Brain to assess if finance principles and promotion gating are accurate and practical for real deployment. Return clear pass/fail checkpoints and threshold improvements. Snapshot: ' + json.dumps(snapshot),
    },
    {
        'check': 'math_brain_reliability_check',
        'question': 'Use Math Brain to validate the quantitative reliability of these governance metrics, including stress-sharpe consistency, score calibration, and potential threshold instability. Snapshot: ' + json.dumps(snapshot),
    },
    {
        'check': 'business_brain_accuracy_optimization',
        'question': 'Use Business Brain to maximize practical financial analysis accuracy. Prioritize model governance, threshold policy tuning, and deployment-readiness tradeoffs with explicit action items. Snapshot: ' + json.dumps(snapshot),
    },
]

rows = []
for q in queries:
    item = {'check': q['check'], 'status': None}
    try:
        r = requests.post(
            f'{BASE_URL}/v1/query',
            json={'question': q['question'], 'k': 8, 'cloud_rerank': False},
            headers=headers,
            timeout=180,
        )
        item['status'] = r.status_code
        if r.status_code == 200:
            j = r.json()
            item['mode'] = j.get('mode')
            item['confidence'] = j.get('confidence')
            item['confidence_label'] = j.get('confidence_label')
            item['selected_modules'] = ', '.join(j.get('selected_modules', []) or [])
            item['answer'] = j.get('answer', '')
            item['context_count'] = len(j.get('context', []))
        else:
            item['mode'] = None
            item['confidence'] = None
            item['confidence_label'] = None
            item['selected_modules'] = ''
            item['answer'] = r.text[:800]
            item['context_count'] = None
    except requests.exceptions.Timeout:
        item['mode'] = None
        item['confidence'] = None
        item['confidence_label'] = None
        item['selected_modules'] = ''
        item['answer'] = 'timeout while waiting for /v1/query response'
        item['context_count'] = None
        item['status'] = 598
    except Exception as e:
        item['mode'] = None
        item['confidence'] = None
        item['confidence_label'] = None
        item['selected_modules'] = ''
        item['answer'] = f'error: {e}'
        item['context_count'] = None
        item['status'] = 599
    rows.append(item)

api_df = pd.DataFrame(rows)
api_out = OUT_DIR / 'api_logic_checks_report.csv'
api_df.to_csv(api_out, index=False)
display(api_df[['check','status','mode','confidence','confidence_label','context_count','selected_modules']])
print('Saved API logic checks:', api_out)
for _, row in api_df.iterrows():
    print('\n' + '=' * 80)
    print('check:', row['check'])
    print('status:', row['status'])
    print('answer:\n', row['answer'])

API key source: C:\Users\dunnm\Downloads\Master-Brain-API\.env::BRIDGE_API_KEY
API key (masked): mast...al | length: 25
health_status: 200
health_json: {'status': 'ok', 'bridge_host': '127.0.0.1', 'bridge_port': 8787, 'api_key_required': True, 'default_index': 'C:\\Users\\dunnm\\Downloads\\Master-Brain-API\\data\\master_brain_index.pkl', 'cwd': 'C:\\Users\\dunnm\\Downloads\\Master-Brain-API'}


,check,status,mode,confidence,confidence_label,context_count,selected_modules
0,business_brain_finance_principles,200,symbolic,0.418906,low,8,"business_brain, cs_brain"
1,math_brain_reliability_check,200,explanation,0.560514,medium,8,"business_brain, cs_brain"
2,business_brain_accuracy_optimization,200,explanation,0.531084,medium,8,"business_brain, cs_brain"


Saved API logic checks: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain\api_logic_checks_report.csv

check: business_brain_finance_principles
status: 200
answer:
 Use the retrieved mathematical context below to reason step-by-step. Validate each algebraic transformation before concluding.

Context:
[0.625] C:\Users\dunnm\Downloads\Master Brain\Business Brain\Accounting and General Business\operations-management-12ed-jay-heizer-pdfdrive-.pdf (page 338)
4 ) Build-to-order (BTO) (p. 285 ) Postponement (p. 285 ) Crossover chart (p. 286 ) Flexibility (p. 288 ) Flowchart (p. 289 ) Time-function mapping (or process mapping) (p. 289 ) Process charts (p. 289 ) Value-stream mapping (VSM) (p. 290 ) Service blueprinting (p. 292 ) Computer numerical control (CNC) (p. 295 ) Additive manufacturing (3D Printing) (p. 295 ) Automatic identification system (AIS) (p. 295 ) Radio frequency identification (RFID) (p. 295 ) Process control (p. 295 ) Vision systems (p. 296 ) Robot (p. 296 )

## Probing Pass: ML + Parallel Optimization + Business Brain Knowledge Sweep

This section performs a deeper algorithm improvement pass:

1. **Parallel policy probing** across reliability, stress, and ML quality gates.
2. **ML-style candidate uplift ranking** to identify near-promote strategies.
3. **Business Brain deep sweep** using econometrics/economics/fintech/quant/TA/time-series perspectives from the indexed knowledge base.

In [18]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import itertools
import numpy as np

gate_probe = pd.read_csv(OUT_DIR / 'promotion_gate_report.csv').copy()

num_cols = [
    'reliability_score',
    'ml_quality_score',
    'promotion_score',
    'stress_sharpe_bps20',
    'stress_total_return_bps20',
    'n_principle_flags',
]
for c in num_cols:
    if c in gate_probe.columns:
        gate_probe[c] = pd.to_numeric(gate_probe[c], errors='coerce')

if 'cv_present' in gate_probe.columns:
    gate_probe['cv_present'] = gate_probe['cv_present'].fillna(False).astype(bool)
else:
    gate_probe['cv_present'] = False

def _evaluate_policy(min_rel: float, min_stress_sharpe: float, min_mlq: float, max_flags: int = 0):
    df = gate_probe.copy()
    pass_fin = (df['reliability_score'] >= min_rel) & (df['n_principle_flags'].fillna(999) <= max_flags)
    pass_stress = (df['stress_total_return_bps20'] > 0) & (df['stress_sharpe_bps20'] >= min_stress_sharpe)
    pass_ml = (df['ml_quality_score'] >= min_mlq) & df['cv_present']

    decision = np.where(pass_fin & pass_stress & pass_ml, 'promote', np.where(pass_fin & pass_stress, 'candidate', 'reject'))
    promotes = int((decision == 'promote').sum())
    candidates = int((decision == 'candidate').sum())

    near_promote_mask = (decision == 'candidate') & (df['ml_quality_score'] < min_mlq)
    ml_gap = (min_mlq - df.loc[near_promote_mask, 'ml_quality_score']).clip(lower=0)
    near_promote_avg_ml_gap = float(ml_gap.mean()) if len(ml_gap) else 0.0

    objective = (promotes * 5.0) + (candidates * 1.5) - (near_promote_avg_ml_gap * 10.0)
    return {
        'min_reliability': min_rel,
        'min_stress_sharpe': min_stress_sharpe,
        'min_ml_quality': min_mlq,
        'promotes': promotes,
        'candidates': candidates,
        'rejects': int((decision == 'reject').sum()),
        'near_promote_avg_ml_gap': near_promote_avg_ml_gap,
        'objective': objective,
    }

rel_grid = [0.45, 0.50, 0.55, 0.60]
stress_grid = [0.20, 0.30, 0.40, 0.50]
mlq_grid = [0.35, 0.40, 0.45, 0.50, 0.55]
combos = list(itertools.product(rel_grid, stress_grid, mlq_grid))

rows = []
with ThreadPoolExecutor(max_workers=min(16, len(combos))) as ex:
    fut_map = {ex.submit(_evaluate_policy, a, b, c): (a, b, c) for a, b, c in combos}
    for fut in as_completed(fut_map):
        rows.append(fut.result())

probe_df = pd.DataFrame(rows).sort_values(['objective', 'promotes', 'candidates'], ascending=False).reset_index(drop=True)
probe_out = OUT_DIR / 'policy_probe_parallel_report.csv'
probe_df.to_csv(probe_out, index=False)

# ML-style uplift targeting: rank candidate runs by smallest gap to promotion thresholds
best_policy = probe_df.iloc[0].to_dict()
rel_t = float(best_policy['min_reliability'])
stress_t = float(best_policy['min_stress_sharpe'])
mlq_t = float(best_policy['min_ml_quality'])

cand = gate_probe.copy()
cand['rel_gap'] = (rel_t - cand['reliability_score']).clip(lower=0)
cand['stress_gap'] = (stress_t - cand['stress_sharpe_bps20']).clip(lower=0)
cand['mlq_gap'] = (mlq_t - cand['ml_quality_score']).clip(lower=0)
cand['flags_gap'] = cand['n_principle_flags'].clip(lower=0)

# lower is better (closer to promotion)
cand['uplift_distance'] = (
    0.45 * cand['rel_gap'].fillna(1.0) +
    0.25 * cand['stress_gap'].fillna(1.0) +
    0.25 * cand['mlq_gap'].fillna(1.0) +
    0.05 * cand['flags_gap'].fillna(1.0)
    )

uplift_cols = [
    'run', 'decision', 'promotion_score', 'reliability_score', 'stress_sharpe_bps20',
    'ml_quality_score', 'n_principle_flags', 'rel_gap', 'stress_gap', 'mlq_gap', 'uplift_distance'
    ]
uplift_df = cand[uplift_cols].sort_values(['uplift_distance', 'promotion_score'], ascending=[True, False]).head(30).reset_index(drop=True)
uplift_out = OUT_DIR / 'candidate_uplift_rankings.csv'
uplift_df.to_csv(uplift_out, index=False)

print('Saved policy probe:', probe_out)
print('Saved candidate uplift ranking:', uplift_out)
print('Best policy from parallel probe:', best_policy)
display(probe_df.head(15))
display(uplift_df.head(20))

Saved policy probe: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain\policy_probe_parallel_report.csv
Saved candidate uplift ranking: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain\candidate_uplift_rankings.csv
Best policy from parallel probe: {'min_reliability': 0.45, 'min_stress_sharpe': 0.2, 'min_ml_quality': 0.35, 'promotes': 0.0, 'candidates': 30.0, 'rejects': 266.0, 'near_promote_avg_ml_gap': 0.34999999999999987, 'objective': 41.5}


C:\Users\dunnm\AppData\Local\Temp\ipykernel_115408\3404445761.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gate_probe['cv_present'] = gate_probe['cv_present'].fillna(False).astype(bool)


,min_reliability,min_stress_sharpe,min_ml_quality,promotes,candidates,rejects,near_promote_avg_ml_gap,objective
0,0.45,0.2,0.35,0,30,266,0.35,41.5
1,0.45,0.2,0.40,0,30,266,0.40,41.0
2,0.45,0.2,0.45,0,30,266,0.45,40.5
3,0.45,0.2,0.50,0,30,266,0.50,40.0
4,0.45,0.2,0.55,0,30,266,0.55,39.5
5,0.45,0.3,0.35,0,28,268,0.35,38.5
6,0.45,0.3,0.40,0,28,268,0.40,38.0
7,0.45,0.3,0.45,0,28,268,0.45,37.5
8,0.45,0.3,0.50,0,28,268,0.50,37.0
9,0.45,0.3,0.55,0,28,268,0.55,36.5


,run,decision,promotion_score,reliability_score,stress_sharpe_bps20,ml_quality_score,n_principle_flags,rel_gap,stress_gap,mlq_gap,uplift_distance
0,bt_crypto_5y_SOL-USD_rf_wv_dwt_db4,candidate,0.475828,0.730540,0.987084,0.0,0,0.0,0.0,0.35,0.0875
1,bt_eurusd_sweep_EURUSD=X_hgb_baseline,reject,0.471614,0.584752,5.508692,0.0,0,0.0,0.0,0.35,0.0875
2,bt_eurusd_sweep_EURUSD=X_mlp_baseline,reject,0.471614,0.584752,5.508692,0.0,0,0.0,0.0,0.35,0.0875
3,bt_eurusd_sweep_EURUSD=X_hgb_stops,reject,0.466018,0.574578,3.217310,0.0,0,0.0,0.0,0.35,0.0875
4,bt_eurusd_sweep_EURUSD=X_mlp_stops,reject,0.466018,0.574578,3.217310,0.0,0,0.0,0.0,0.35,0.0875
5,bt_crypto_5y_ETH-USD_hgb_wv_dwt_db6,reject,0.372346,0.530520,1.074130,0.0,0,0.0,0.0,0.35,0.0875
6,bt_crypto_5y_SOL-USD_hgb_wv_dwt_db4,reject,0.370906,0.625304,0.359852,0.0,0,0.0,0.0,0.35,0.0875
7,bt_crypto_5y_SOL-USD_mlp_wv_dwt_db4,reject,0.363291,0.595618,0.476007,0.0,0,0.0,0.0,0.35,0.0875
8,bt_crypto_5y_SOL-USD_rf_proba_hmm_wavelet,reject,0.346818,0.551510,0.579826,0.0,0,0.0,0.0,0.35,0.0875
9,bt_crypto_5y_ETH-USD_mlp_wv_dwt_db4,reject,0.344384,0.535523,0.664620,0.0,0,0.0,0.0,0.35,0.0875


In [19]:
# Deep Business Brain sweep across econometrics/economics/fintech/quant/TA/time-series books
deep_sweep_question = (
    'Use Business Brain for a thorough algorithm improvement sweep. '
    'Ground recommendations in econometrics, economics, fintech, quantitative and automated trading, '
    'technical analysis, and time-series/forecasting literature from the indexed books. '
    'Given this snapshot, provide: '
    '(1) the top 10 upgrade actions ranked by business impact and implementation effort, '
    '(2) concrete threshold policy updates for promotion gating, '
    '(3) candidate rescue plan for near-promote runs, '
    '(4) risk controls to avoid overfitting and unstable live deployment, '
    '(5) a 30/60/90-day execution roadmap. Snapshot: ' + json.dumps(snapshot)
    )

deep_payload = {'question': deep_sweep_question, 'k': 12, 'cloud_rerank': False}
deep_item = {'check': 'business_brain_deep_sweep', 'status': None}

try:
    deep_r = requests.post(f'{BASE_URL}/v1/query', json=deep_payload, headers=headers, timeout=240)
    deep_item['status'] = deep_r.status_code
    if deep_r.status_code == 200:
        deep_j = deep_r.json()
        deep_item['mode'] = deep_j.get('mode')
        deep_item['confidence'] = deep_j.get('confidence')
        deep_item['confidence_label'] = deep_j.get('confidence_label')
        deep_item['selected_modules'] = ', '.join(deep_j.get('selected_modules', []) or [])
        deep_item['answer'] = deep_j.get('answer', '')
        deep_item['context_count'] = len(deep_j.get('context', []))
    else:
        deep_item['mode'] = None
        deep_item['confidence'] = None
        deep_item['confidence_label'] = None
        deep_item['selected_modules'] = ''
        deep_item['answer'] = deep_r.text[:2000]
        deep_item['context_count'] = None
except requests.exceptions.Timeout:
    deep_item['mode'] = None
    deep_item['confidence'] = None
    deep_item['confidence_label'] = None
    deep_item['selected_modules'] = ''
    deep_item['answer'] = 'timeout while waiting for deep Business Brain sweep response'
    deep_item['context_count'] = None
    deep_item['status'] = 598
except Exception as e:
    deep_item['mode'] = None
    deep_item['confidence'] = None
    deep_item['confidence_label'] = None
    deep_item['selected_modules'] = ''
    deep_item['answer'] = f'error: {e}'
    deep_item['context_count'] = None
    deep_item['status'] = 599

deep_df = pd.DataFrame([deep_item])
deep_out = OUT_DIR / 'business_brain_deep_sweep.csv'
deep_df.to_csv(deep_out, index=False)

print('Saved deep sweep:', deep_out)
display(deep_df[['check','status','mode','confidence','confidence_label','context_count','selected_modules']])
print('\n' + '=' * 80)
print('business_brain_deep_sweep status:', deep_item['status'])
print('answer:\n', deep_item['answer'])

Saved deep sweep: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain\business_brain_deep_sweep.csv


,check,status,mode,confidence,confidence_label,context_count,selected_modules
0,business_brain_deep_sweep,200,symbolic,0.470599,low,12,"business_brain, cs_brain"



business_brain_deep_sweep status: 200
answer:
 Use the retrieved mathematical context below to reason step-by-step. Validate each algebraic transformation before concluding.

Context:
[0.698] C:\Users\dunnm\Downloads\Master Brain\Computer Science Brain\Designing Data-Intensive Applications The Big Ideas Behind Reliable, Scalable, and Maintainable Systems by Martin Kleppmann (z-lib.org).pdf (page 351)
ution (see “Actual Serial Execution” on page 252) are typically linearizable. However, serializable snapshot isolation (see “Serializable Snapshot Isolation (SSI)” on page 261) is not linearizable: by design, it makes reads from a consistent snapshot, to avoid lock contention between readers and writers. The whole point of a consistent snapshot is that it does not include writes that are more recent than the snapshot, and thus reads from the snapshot are not linearizable. Linearizability | 329

[0.633] C:\Users\dunnm\Downloads\Master Brain\Business Brain\Accounting and General Business\op

In [20]:
# Business Brain fallback sweep (force practical narrative output)
fallback_question = (
    'Business Brain only. Return practical business and quantitative trading guidance in plain language (no symbolic math). '
    'Use econometrics, economics, fintech, quantitative/automated trading, technical analysis, and time-series forecasting insights '
    'from available indexed books. Provide: top 10 improvements, gate-threshold redesign, ML feature engineering upgrades, '
    'parallel computing recommendations, and risk governance guardrails. Include a concise rationale for each recommendation and '
    'cite which knowledge domains/books informed it. Snapshot: ' + json.dumps(snapshot)
    )

fb_payload = {'question': fallback_question, 'k': 12, 'cloud_rerank': False}
fb_item = {'check': 'business_brain_fallback_sweep', 'status': None}

try:
    fb_r = requests.post(f'{BASE_URL}/v1/query', json=fb_payload, headers=headers, timeout=240)
    fb_item['status'] = fb_r.status_code
    if fb_r.status_code == 200:
        fb_j = fb_r.json()
        fb_item['mode'] = fb_j.get('mode')
        fb_item['confidence'] = fb_j.get('confidence')
        fb_item['confidence_label'] = fb_j.get('confidence_label')
        fb_item['selected_modules'] = ', '.join(fb_j.get('selected_modules', []) or [])
        fb_item['answer'] = fb_j.get('answer', '')
        fb_item['context_count'] = len(fb_j.get('context', []))
    else:
        fb_item['mode'] = None
        fb_item['confidence'] = None
        fb_item['confidence_label'] = None
        fb_item['selected_modules'] = ''
        fb_item['answer'] = fb_r.text[:2000]
        fb_item['context_count'] = None
except requests.exceptions.Timeout:
    fb_item['mode'] = None
    fb_item['confidence'] = None
    fb_item['confidence_label'] = None
    fb_item['selected_modules'] = ''
    fb_item['answer'] = 'timeout while waiting for Business Brain fallback response'
    fb_item['context_count'] = None
    fb_item['status'] = 598
except Exception as e:
    fb_item['mode'] = None
    fb_item['confidence'] = None
    fb_item['confidence_label'] = None
    fb_item['selected_modules'] = ''
    fb_item['answer'] = f'error: {e}'
    fb_item['context_count'] = None
    fb_item['status'] = 599

fb_df = pd.DataFrame([fb_item])
fb_out = OUT_DIR / 'business_brain_fallback_sweep.csv'
fb_df.to_csv(fb_out, index=False)

print('Saved fallback sweep:', fb_out)
display(fb_df[['check','status','mode','confidence','confidence_label','context_count','selected_modules']])
print('\n' + '=' * 80)
print('business_brain_fallback_sweep status:', fb_item['status'])
print('answer:\n', fb_item['answer'])

Saved fallback sweep: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain\business_brain_fallback_sweep.csv


,check,status,mode,confidence,confidence_label,context_count,selected_modules
0,business_brain_fallback_sweep,200,symbolic,0.465311,low,12,"business_brain, cs_brain"



business_brain_fallback_sweep status: 200
answer:
 Use the retrieved mathematical context below to reason step-by-step. Validate each algebraic transformation before concluding.

Context:
[0.700] C:\Users\dunnm\Downloads\Master Brain\Business Brain\Technical Analysis\4c7037365a4bf1623734c1c899baed7855061ace.pdf (page 23)
P1: JYS c01 JWBK321-Chan September 24, 2008 13:44 Printer: Yet to come CHAPTER 1 The Whats, Whos, and Whys of Quantitative Trading I f you are curious enough to pick up this book, you probably have already heard of quantitative trading. But even for readers who learned about this kind of trading from the mainstream media before, it is worth clearing up some common misconceptions. Quantitative trading, also known as algorithmic trading, is the trading of securities based strictly on the buy/sell decisions of com- puter algorithms. The computer algorithms are designed and per- haps programmed by the traders themselves, based on the histor- ical performance of the encode

In [21]:
# Business Brain explanation-mode attempt for actionable roadmap
explain_question = (
    'Explain clearly and accurately using the indexed finance and quant books. '
    'Provide a practical optimization plan for this trading governance pipeline: '
    'top 10 algorithm changes, threshold tuning, econometric validation, time-series improvements, '
    'feature engineering priorities, and parallel processing architecture upgrades. '
    'Use concise bullets and prioritize by impact. Snapshot: ' + json.dumps(snapshot)
    )

explain_item = {'check': 'business_brain_explain_sweep', 'status': None}
try:
    explain_r = requests.post(
        f'{BASE_URL}/v1/query',
        json={'question': explain_question, 'k': 12, 'cloud_rerank': False},
        headers=headers,
        timeout=240,
    )
    explain_item['status'] = explain_r.status_code
    if explain_r.status_code == 200:
        explain_j = explain_r.json()
        explain_item['mode'] = explain_j.get('mode')
        explain_item['confidence'] = explain_j.get('confidence')
        explain_item['confidence_label'] = explain_j.get('confidence_label')
        explain_item['selected_modules'] = ', '.join(explain_j.get('selected_modules', []) or [])
        explain_item['answer'] = explain_j.get('answer', '')
        explain_item['context_count'] = len(explain_j.get('context', []))
    else:
        explain_item['mode'] = None
        explain_item['confidence'] = None
        explain_item['confidence_label'] = None
        explain_item['selected_modules'] = ''
        explain_item['answer'] = explain_r.text[:2000]
        explain_item['context_count'] = None
except requests.exceptions.Timeout:
    explain_item['mode'] = None
    explain_item['confidence'] = None
    explain_item['confidence_label'] = None
    explain_item['selected_modules'] = ''
    explain_item['answer'] = 'timeout while waiting for Business Brain explanation sweep response'
    explain_item['context_count'] = None
    explain_item['status'] = 598
except Exception as e:
    explain_item['mode'] = None
    explain_item['confidence'] = None
    explain_item['confidence_label'] = None
    explain_item['selected_modules'] = ''
    explain_item['answer'] = f'error: {e}'
    explain_item['context_count'] = None
    explain_item['status'] = 599

explain_df = pd.DataFrame([explain_item])
explain_out = OUT_DIR / 'business_brain_explain_sweep.csv'
explain_df.to_csv(explain_out, index=False)

print('Saved explain sweep:', explain_out)
display(explain_df[['check','status','mode','confidence','confidence_label','context_count','selected_modules']])
print('\n' + '=' * 80)
print('business_brain_explain_sweep status:', explain_item['status'])
print('answer:\n', explain_item['answer'])

Saved explain sweep: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain\business_brain_explain_sweep.csv


,check,status,mode,confidence,confidence_label,context_count,selected_modules
0,business_brain_explain_sweep,200,symbolic,0.41909,low,12,"business_brain, cs_brain"



business_brain_explain_sweep status: 200
answer:
 Use the retrieved mathematical context below to reason step-by-step. Validate each algebraic transformation before concluding.

Context:
[0.628] C:\Users\dunnm\Downloads\Master Brain\Business Brain\Accounting and General Business\operations-management-12ed-jay-heizer-pdfdrive-.pdf (page 338)
4 ) Build-to-order (BTO) (p. 285 ) Postponement (p. 285 ) Crossover chart (p. 286 ) Flexibility (p. 288 ) Flowchart (p. 289 ) Time-function mapping (or process mapping) (p. 289 ) Process charts (p. 289 ) Value-stream mapping (VSM) (p. 290 ) Service blueprinting (p. 292 ) Computer numerical control (CNC) (p. 295 ) Additive manufacturing (3D Printing) (p. 295 ) Automatic identification system (AIS) (p. 295 ) Radio frequency identification (RFID) (p. 295 ) Process control (p. 295 ) Vision systems (p. 296 ) Robot (p. 296 ) Automated storage and retrieval system (ASRS) (p. 296 ) Automated guided vehicle (AGV) (p. 296 ) Flexible manufacturing system (FMS

In [22]:
# Auto-build a concrete algorithm improvement roadmap from probe outputs
probe_df = pd.read_csv(OUT_DIR / 'policy_probe_parallel_report.csv')
uplift_df = pd.read_csv(OUT_DIR / 'candidate_uplift_rankings.csv')
quality_df = pd.read_csv(OUT_DIR / 'quality_metrics_report.csv')
mon_df = pd.read_csv(OUT_DIR / 'monitoring_metrics_report.csv')

q = dict(zip(quality_df['metric'], quality_df['value']))
m = dict(zip(mon_df['metric'], mon_df['value']))
best = probe_df.iloc[0].to_dict()

top_runs = uplift_df.head(10)['run'].tolist()
top_runs_md = '\n'.join([f'- `{r}`' for r in top_runs])

lines = []
lines.append('# Algorithm Improvement Roadmap (Probing Pass)')
lines.append('')
lines.append('## Current bottleneck diagnosis')
lines.append('')
lines.append(f"- Promoted runs: **{int(m.get('runs_promote', 0))}**")
lines.append(f"- Candidate runs: **{int(m.get('runs_candidate', 0))}**")
lines.append(f"- Reject runs: **{int(m.get('runs_reject', 0))}**")
lines.append(f"- Mean ML quality score: **{float(q.get('ml_quality_mean', float('nan'))):.4f}**")
lines.append(f"- Pass-ML rate: **{float(q.get('pass_ml_rate', float('nan'))):.4f}**")
lines.append(f"- Mean stress Sharpe (20 bps): **{float(q.get('stress_sharpe_bps20_mean', float('nan'))):.4f}**")
lines.append('')
lines.append('## Best near-term policy profile from parallel sweep')
lines.append('')
lines.append(f"- min_reliability: **{best['min_reliability']}**")
lines.append(f"- min_stress_sharpe: **{best['min_stress_sharpe']}**")
lines.append(f"- min_ml_quality: **{best['min_ml_quality']}**")
lines.append(f"- Expected candidates under this profile: **{int(best['candidates'])}**")
lines.append(f"- Near-promote average ML gap: **{best['near_promote_avg_ml_gap']:.4f}**")
lines.append('')
lines.append('## Top 10 upgrade actions (impact-ordered)')
lines.append('')
lines.append('1. **Fix ML-quality gate bottleneck**: add calibrated probability outputs + CV completeness checks for every model run.')
lines.append('2. **Raise CV coverage to 100%**: enforce minimum fold coverage and fail closed when `cv_metrics.csv` is missing.')
lines.append('3. **Improve feature stack**: add regime, volatility, spread, and higher-moment features with rolling stability diagnostics.')
lines.append('4. **Econometric robustness**: add walk-forward and purged time-series CV to reduce leakage and inflation of in-sample quality.')
lines.append('5. **Time-series forecasting upgrades**: compare baseline vs AR/ARX/state-space/sequence models with identical transaction-cost assumptions.')
lines.append('6. **Technical-analysis signal hygiene**: keep only signals with out-of-sample incremental information value and low collinearity.')
lines.append('7. **Risk-first deployment gates**: require positive stressed return and stress Sharpe under cost/slippage perturbation scenarios.')
lines.append('8. **Parallel compute architecture**: execute model/backtest grid in process-level pools and shard by asset/family for faster iteration.')
lines.append('9. **Business governance layer**: add scorecards (expected capacity, turnover constraints, implementation friction) before promotion.')
lines.append('10. **Automated rescue loop**: retrain top near-promote runs with targeted ML-gap closure and re-gate automatically.')
lines.append('')
lines.append('## Near-promote rescue candidates')
lines.append('')
lines.append(top_runs_md)

roadmap_out = OUT_DIR / 'algorithm_improvement_roadmap.md'
roadmap_out.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('Saved roadmap:', roadmap_out)
print('\n'.join(lines[:35]))

Saved roadmap: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain\algorithm_improvement_roadmap.md
# Algorithm Improvement Roadmap (Probing Pass)

## Current bottleneck diagnosis

- Promoted runs: **0**
- Candidate runs: **1**
- Reject runs: **295**
- Mean ML quality score: **0.4727**
- Pass-ML rate: **0.0000**
- Mean stress Sharpe (20 bps): **-0.8877**

## Best near-term policy profile from parallel sweep

- min_reliability: **0.45**
- min_stress_sharpe: **0.2**
- min_ml_quality: **0.35**
- Expected candidates under this profile: **30**
- Near-promote average ML gap: **0.3500**

## Top 10 upgrade actions (impact-ordered)

1. **Fix ML-quality gate bottleneck**: add calibrated probability outputs + CV completeness checks for every model run.
2. **Raise CV coverage to 100%**: enforce minimum fold coverage and fail closed when `cv_metrics.csv` is missing.
3. **Improve feature stack**: add regime, volatility, spread, and higher-moment features with rolling stability diagno

## Automated Rescue Loop (Top Near-Promote Runs)

This section automatically:

1. Selects top near-promote runs from `candidate_uplift_rankings.csv`.
2. Reconstructs each run's source input/variant from its `backtest_summary.json`.
3. Re-runs targeted `strategy_sweep.py` variants in parallel into `artifacts/backtests` with a `bt_rescue` prefix.
4. Re-runs Master Brain governance scoring and compares before/after candidate/promote counts.

In [24]:
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
import shlex

TOP_N_RESCUE = 10
MAX_PARALLEL_RESCUE = 4

uplift_path = OUT_DIR / 'candidate_uplift_rankings.csv'
uplift_df = pd.read_csv(uplift_path).head(TOP_N_RESCUE).copy()
candidate_runs = uplift_df['run'].tolist()

def _resolve_input_path(raw_input: str) -> Path | None:
    if not raw_input:
        return None
    p = Path(raw_input)
    candidates = [
        ROOT / p,
        ROOT / 'artifacts' / 'outputs' / p,
        ROOT / 'artifacts' / p,
    ]
    for c in candidates:
        if c.exists() and c.is_file():
            return c.resolve()
    target_name = p.name
    for c in (ROOT / 'artifacts').rglob(target_name):
        if c.is_file():
            return c.resolve()
    return None

meta_rows = []
for run in candidate_runs:
    summary_json = ROOT / 'artifacts' / 'backtests' / run / 'backtest_summary.json'
    if not summary_json.exists():
        continue
    s = json.loads(summary_json.read_text(encoding='utf-8'))
    raw_input = s.get('input')
    variant = s.get('variant')
    asset = s.get('asset')
    in_csv = _resolve_input_path(raw_input)
    if variant and in_csv is not None:
        meta_rows.append({
            'run': run,
            'asset': asset,
            'variant': variant,
            'input_raw': raw_input,
            'in_csv': str(in_csv),
        })

meta_df = pd.DataFrame(meta_rows)
if meta_df.empty:
    raise RuntimeError('No resolvable candidate metadata found for rescue loop.')

groups = defaultdict(lambda: {'variants': set(), 'assets': set()})
for r in meta_rows:
    groups[r['in_csv']]['variants'].add(r['variant'])
    groups[r['in_csv']]['assets'].add(r['asset'] or 'UNKNOWN')

strategy_sweep_py = ROOT / 'projects' / 'strategy_sweep.py'
assert strategy_sweep_py.exists(), f'Missing script: {strategy_sweep_py}'

rescue_root = ROOT / 'artifacts' / 'backtests'
rescue_root.mkdir(parents=True, exist_ok=True)

def _run_rescue_group(in_csv: str, variants: list[str]) -> dict:
    cores = sorted({v.split('_', 1)[0] for v in variants if '_' in v})
    cmd = [
        sys.executable, '-m', 'projects.strategy_sweep',
        '--in-csv', in_csv,
        '--out-dir', str(rescue_root),
        '--out-prefix', 'bt_rescue',
        '--cores', ','.join(cores) if cores else 'hgb,rf,mlp',
        '--only-variants', ','.join(sorted(set(variants))),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd=str(ROOT))
    return {
        'in_csv': in_csv,
        'variants': ','.join(sorted(set(variants))),
        'return_code': proc.returncode,
        'stdout_tail': proc.stdout[-3000:],
        'stderr_tail': proc.stderr[-3000:],
        'cmd': ' '.join(shlex.quote(x) for x in cmd),
    }

jobs = [(k, sorted(v['variants'])) for k, v in groups.items()]
results = []
with ThreadPoolExecutor(max_workers=min(MAX_PARALLEL_RESCUE, len(jobs))) as ex:
    futs = [ex.submit(_run_rescue_group, in_csv, variants) for in_csv, variants in jobs]
    for fut in as_completed(futs):
        results.append(fut.result())

rescue_exec_df = pd.DataFrame(results).sort_values(['return_code','in_csv']).reset_index(drop=True)
rescue_exec_out = OUT_DIR / 'rescue_execution_report.csv'
rescue_exec_df.to_csv(rescue_exec_out, index=False)

display(meta_df)
display(rescue_exec_df[['in_csv','variants','return_code']])
print('Saved rescue execution report:', rescue_exec_out)
if (rescue_exec_df['return_code'] != 0).any():
    print('Some rescue jobs failed. Review stderr_tail in rescue_execution_report.csv')

,run,asset,variant,input_raw,in_csv
0,bt_eurusd_sweep_EURUSD=X_hgb_baseline,EURUSD=X,hgb_baseline,smoke_market_fx_1mo_1d\EURUSD=X_1mo_1d.csv,C:\Users\dunnm\source\repos\finance-scrapers-p...
1,bt_eurusd_sweep_EURUSD=X_mlp_baseline,EURUSD=X,mlp_baseline,smoke_market_fx_1mo_1d\EURUSD=X_1mo_1d.csv,C:\Users\dunnm\source\repos\finance-scrapers-p...
2,bt_eurusd_sweep_EURUSD=X_hgb_stops,EURUSD=X,hgb_stops,smoke_market_fx_1mo_1d\EURUSD=X_1mo_1d.csv,C:\Users\dunnm\source\repos\finance-scrapers-p...
3,bt_eurusd_sweep_EURUSD=X_mlp_stops,EURUSD=X,mlp_stops,smoke_market_fx_1mo_1d\EURUSD=X_1mo_1d.csv,C:\Users\dunnm\source\repos\finance-scrapers-p...


,in_csv,variants,return_code
0,C:\Users\dunnm\source\repos\finance-scrapers-p...,"hgb_baseline,hgb_stops,mlp_baseline,mlp_stops",0


Saved rescue execution report: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain\rescue_execution_report.csv


In [25]:
# Re-score governance after rescue runs and compare before/after
rescue_out_dir = ROOT / 'master_brain_rescue'
rescue_out_dir.mkdir(parents=True, exist_ok=True)

rescore_cmd = [
    sys.executable,
    str(PIPELINE),
    '--in-root', str(ROOT),
    '--out-dir', str(rescue_out_dir),
    '--gate-stress-bps', '20',
    '--stress-bps-grid', '10,20,35',
]

rescore = subprocess.run(rescore_cmd, capture_output=True, text=True, cwd=str(ROOT))
print('rescore return_code:', rescore.returncode)
print(rescore.stdout[-3000:])
if rescore.returncode != 0:
    print('stderr_tail:')
    print(rescore.stderr[-3000:])
    raise RuntimeError('Rescore after rescue failed')

before_summary = json.loads((OUT_DIR / 'master_brain_summary.json').read_text(encoding='utf-8'))
after_summary = json.loads((rescue_out_dir / 'master_brain_summary.json').read_text(encoding='utf-8'))

before_counts = before_summary.get('counts', {})
after_counts = after_summary.get('counts', {})

cmp = pd.DataFrame([
    {'metric': 'backtests_scored', 'before': before_counts.get('backtests_scored'), 'after': after_counts.get('backtests_scored')},
    {'metric': 'promoted_runs', 'before': before_counts.get('promoted_runs'), 'after': after_counts.get('promoted_runs')},
    {'metric': 'ml_runs_scored', 'before': before_counts.get('ml_runs_scored'), 'after': after_counts.get('ml_runs_scored')},
])
cmp['delta'] = pd.to_numeric(cmp['after'], errors='coerce') - pd.to_numeric(cmp['before'], errors='coerce')

cmp_out = rescue_out_dir / 'rescue_before_after_comparison.csv'
cmp.to_csv(cmp_out, index=False)

report_lines = [
    '# Rescue Loop Outcome Summary',
    '',
    f"- Rescue jobs executed: **{len(rescue_exec_df)}**",
    f"- Rescue job failures: **{int((rescue_exec_df['return_code'] != 0).sum())}**",
    f"- Backtests scored before: **{before_counts.get('backtests_scored', 0)}**",
    f"- Backtests scored after: **{after_counts.get('backtests_scored', 0)}**",
    f"- Promoted runs before: **{before_counts.get('promoted_runs', 0)}**",
    f"- Promoted runs after: **{after_counts.get('promoted_runs', 0)}**",
    '',
    '## Next tuning focus',
    '',
    '- If promotes remain 0, prioritize ML quality uplift (CV completeness + calibration) for rescued variants.',
    '- Iterate with lower-variance features and stronger probability calibration, then rerun this rescue loop.',
    '- Keep stress and risk gates strict while lifting model-quality reliability.',
    '',
    f"- Comparison CSV: `{cmp_out}`",
    f"- Rescue execution CSV: `{rescue_exec_out}`",
]

rescue_report_md = rescue_out_dir / 'rescue_loop_outcome.md'
rescue_report_md.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')

display(cmp)
print('Saved comparison:', cmp_out)
print('Saved rescue summary:', rescue_report_md)

rescore return_code: 0
Wrote: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain_rescue\finance_principles_report.csv
Wrote: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain_rescue\ml_principles_report.csv
Wrote: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain_rescue\promotion_gate_report.csv
Wrote: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain_rescue\monitoring_metrics_report.csv
Wrote: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain_rescue\folder_cleanup_plan.csv
Wrote: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain_rescue\folder_cleanup_execution_report.csv
Wrote: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain_rescue\folder_cleanup_remaining_plan.csv
Wrote: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain_rescue\quality_metrics_report.csv
Wrote: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain_rescue\master_

,metric,before,after,delta
0,backtests_scored,296,300,4
1,promoted_runs,0,0,0
2,ml_runs_scored,6,6,0


Saved comparison: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain_rescue\rescue_before_after_comparison.csv
Saved rescue summary: C:\Users\dunnm\source\repos\finance-scrapers-portfolio\master_brain_rescue\rescue_loop_outcome.md


## TA/Finance Classifier Mining + Fintech Model Integration (Business Brain)

This section asks Business Brain to specifically mine the **technical analysis** and **finance** sections for classifier-strength signal ideas, then maps those ideas to the existing in-repo fintech model families (HGB/RF/MLP + HMM/FFT/Wavelet/DMD variants).

In [ ]:
# Focused Business Brain mining: TA + Finance -> strong classifier ideas + fintech integration map
if 'snapshot' not in globals():
    summary_local = json.loads((OUT_DIR / 'master_brain_summary.json').read_text(encoding='utf-8'))
    quality_local = pd.read_csv(OUT_DIR / 'quality_metrics_report.csv')
    monitoring_local = pd.read_csv(OUT_DIR / 'monitoring_metrics_report.csv')
    snapshot = {
        'counts': summary_local.get('counts', {}),
        'quality_metrics': dict(zip(quality_local['metric'], quality_local['value'])),
        'monitoring_metrics': dict(zip(monitoring_local['metric'], monitoring_local['value'])),
    }

classifier_queries = [
    {
        'check': 'business_brain_ta_classifier_signals',
        'question': (
            'Read the technical analysis sections and identify signals/features that produce robust classifiers in live trading. '
            'Prioritize trend, momentum, volatility, market-structure, and regime diagnostics. '
            'Return: (1) top 12 classifier-ready features, (2) expected failure modes, '
            '(3) how to validate each with walk-forward CV and calibration checks. Snapshot: ' + json.dumps(snapshot)
        ),
    },
    {
        'check': 'business_brain_finance_classifier_principles',
        'question': (
            'Read the finance sections and identify principles that improve classifier reliability and deployment outcomes. '
            'Return: (1) risk-adjusted objective design, (2) thresholding and class-imbalance policy, '
            '(3) transaction-cost and slippage-aware validation guardrails, '
            '(4) top anti-overfitting controls. Snapshot: ' + json.dumps(snapshot)
        ),
    },
    {
        'check': 'business_brain_fintech_model_integration',
        'question': (
            'Using fintech and quantitative sections, map recommended methods to this model stack: '
            'hgb/rf/mlp cores with hmm/fft/wavelet/dmd gates and probability-based entry variants. '
            'Return a practical integration plan with ranked model combinations and why each should improve promotion odds. '
            'Snapshot: ' + json.dumps(snapshot)
        ),
    },
]

bb_rows = []
for q in classifier_queries:
    row = {'check': q['check'], 'status': None}
    try:
        rr = requests.post(
            f'{BASE_URL}/v1/query',
            json={'question': q['question'], 'k': 12, 'cloud_rerank': False},
            headers=headers,
            timeout=240,
        )
        row['status'] = rr.status_code
        if rr.status_code == 200:
            jj = rr.json()
            row['mode'] = jj.get('mode')
            row['confidence'] = jj.get('confidence')
            row['confidence_label'] = jj.get('confidence_label')
            row['selected_modules'] = ', '.join(jj.get('selected_modules', []) or [])
            row['answer'] = jj.get('answer', '')
            row['context_count'] = len(jj.get('context', []))
        else:
            row['mode'] = None
            row['confidence'] = None
            row['confidence_label'] = None
            row['selected_modules'] = ''
            row['answer'] = rr.text[:3000]
            row['context_count'] = None
    except requests.exceptions.Timeout:
        row['status'] = 598
        row['mode'] = None
        row['confidence'] = None
        row['confidence_label'] = None
        row['selected_modules'] = ''
        row['answer'] = 'timeout while waiting for TA/finance classifier mining response'
        row['context_count'] = None
    except Exception as e:
        row['status'] = 599
        row['mode'] = None
        row['confidence'] = None
        row['confidence_label'] = None
        row['selected_modules'] = ''
        row['answer'] = f'error: {e}'
        row['context_count'] = None
    bb_rows.append(row)

bb_df = pd.DataFrame(bb_rows)
bb_out = OUT_DIR / 'business_brain_ta_finance_fintech_classifier_sweep.csv'
bb_df.to_csv(bb_out, index=False)

# Build a concrete integration map from existing in-repo fintech model families
integration_map = pd.DataFrame([
    {'family': 'trend/regime', 'signals': 'hmm_state, drawdown regime, volatility state', 'existing_variants': 'hgb_hmm, rf_hmm, mlp_hmm, *_hmm_fft_dmd', 'fit_for': 'bull/bear/sideways separation'},
    {'family': 'cycle/spectral', 'signals': 'dominant period, spectral energy concentration', 'existing_variants': 'hgb_fft, rf_fft, mlp_fft, *_wavelet, *_wv_*', 'fit_for': 'swing/mean-reversion timing'},
    {'family': 'dynamics', 'signals': 'DMD growth/eigen-structure', 'existing_variants': 'hgb_dmd, rf_dmd, mlp_dmd, *_hmm_*_dmd', 'fit_for': 'state-transition sensitivity'},
    {'family': 'probability quality', 'signals': 'buy/sell posterior confidence', 'existing_variants': '*_proba_hmm_fft, *_proba_hmm_wavelet', 'fit_for': 'entry selectivity + calibration'},
    {'family': 'risk overlays', 'signals': 'vol target, stop/take-profit', 'existing_variants': '*_voltarget, *_stops', 'fit_for': 'drawdown control and stress resilience'},
])
int_out = OUT_DIR / 'fintech_model_integration_map.csv'
integration_map.to_csv(int_out, index=False)

display(bb_df[['check','status','mode','confidence','confidence_label','context_count','selected_modules']])
display(integration_map)
print('Saved Business Brain classifier sweep:', bb_out)
print('Saved fintech integration map:', int_out)
for _, r in bb_df.iterrows():
    print('\n' + '=' * 80)
    print('check:', r['check'])
    print('status:', r['status'])
    print('answer:\n', r['answer'])

## ML-Quality Rescue Retraining (Walk-Forward + Calibration) and Rescue Rerun

This section trains calibrated walk-forward classifiers for the top near-promote rescue cohort, then reruns the same rescue sweep and links each rerun to the calibrated ML artifact so Master Brain can score ML quality directly in promotion gating.

In [ ]:
# Calibrated walk-forward retraining for top near-promote rescue cohort
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from joblib import dump
from collections import defaultdict

from utils.ml_features import LabelSpec, build_features, make_labels

TOP_N_ML_RESCUE = 12
N_SPLITS = 5
TEST_WINDOW = 20
PURGE = 5

uplift_df_local = pd.read_csv(OUT_DIR / 'candidate_uplift_rankings.csv').head(TOP_N_ML_RESCUE).copy()
candidate_runs_local = uplift_df_local['run'].tolist()


def _load_csv_with_date(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    date_col = None
    for c in ('Date', 'Datetime', 'date', 'datetime'):
        if c in df.columns:
            date_col = c
            break
    if date_col is None and df.columns.size and str(df.columns[0]).lower().startswith('unnamed'):
        date_col = df.columns[0]
    if date_col is not None:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
    return df


def _resolve_input_path(raw_input: str) -> Path | None:
    if not raw_input:
        return None
    p = Path(raw_input)
    candidates = [ROOT / p, ROOT / 'artifacts' / 'outputs' / p, ROOT / 'artifacts' / p]
    for c in candidates:
        if c.exists() and c.is_file():
            return c.resolve()
    target_name = p.name
    for c in (ROOT / 'artifacts').rglob(target_name):
        if c.is_file():
            return c.resolve()
    return None


def _walkforward_splits(n: int, n_splits: int = 5, test_window: int = 20, purge: int = 5, min_train: int = 60):
    first_test_start = max(min_train, purge)
    last_possible_start = n - test_window
    if last_possible_start <= first_test_start:
        return []
    starts = np.linspace(first_test_start, last_possible_start, num=n_splits, dtype=int)
    starts = np.unique(starts)
    out = []
    for s in starts:
        train_end = s - purge
        if train_end <= 0:
            continue
        tr = np.arange(0, train_end)
        te = np.arange(s, min(n, s + test_window))
        if len(tr) >= min_train and len(te) >= 5:
            out.append((tr, te))
    return out


def _make_base(core: str, rs: int):
    c = str(core).lower().strip()
    if c == 'rf':
        model = RandomForestClassifier(
            n_estimators=700,
            max_depth=None,
            min_samples_leaf=3,
            class_weight='balanced_subsample',
            n_jobs=-1,
            random_state=rs,
        )
    elif c == 'mlp':
        model = MLPClassifier(
            hidden_layer_sizes=(128, 64),
            alpha=5e-4,
            learning_rate_init=8e-4,
            max_iter=400,
            early_stopping=True,
            random_state=rs,
        )
    else:
        model = HistGradientBoostingClassifier(
            learning_rate=0.04,
            max_depth=6,
            max_iter=1000,
            random_state=rs,
        )
    return Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', model)])


def _ece_multiclass(y_true: np.ndarray, proba: np.ndarray, bins: int = 10) -> float:
    if proba.ndim != 2 or len(y_true) == 0:
        return float('nan')
    y = np.asarray(y_true).astype(int)
    pmax = proba.max(axis=1)
    pred = proba.argmax(axis=1)
    correct = (pred == y).astype(float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    n = len(y)
    for i in range(bins):
        lo, hi = edges[i], edges[i + 1]
        if i < bins - 1:
            m = (pmax >= lo) & (pmax < hi)
        else:
            m = (pmax >= lo) & (pmax <= hi)
        if not np.any(m):
            continue
        conf = float(pmax[m].mean())
        acc = float(correct[m].mean())
        ece += (m.sum() / n) * abs(acc - conf)
    return float(ece)


meta_rows_local = []
for run in candidate_runs_local:
    s_path = ROOT / 'artifacts' / 'backtests' / run / 'backtest_summary.json'
    if not s_path.exists():
        continue
    s = json.loads(s_path.read_text(encoding='utf-8'))
    variant = str(s.get('variant') or '')
    core = variant.split('_', 1)[0] if '_' in variant else str(s.get('model') or '').strip().lower()
    asset = str(s.get('asset') or 'UNKNOWN')
    in_csv = _resolve_input_path(s.get('input'))
    if in_csv is None or not core:
        continue
    meta_rows_local.append({'run': run, 'asset': asset, 'variant': variant, 'core': core, 'in_csv': str(in_csv)})

meta_df_local = pd.DataFrame(meta_rows_local)
if meta_df_local.empty:
    raise RuntimeError('No candidate metadata resolved for ML rescue retraining.')

group_df = (
    meta_df_local[['asset', 'core', 'in_csv']]
    .drop_duplicates()
    .sort_values(['asset', 'core'])
    .reset_index(drop=True)
)

ml_rescue_root = ROOT / 'artifacts' / 'ml'
ml_rescue_root.mkdir(parents=True, exist_ok=True)

retrain_rows = []
for _, g in group_df.iterrows():
    asset = str(g['asset'])
    core = str(g['core']).lower().strip()
    in_csv = Path(g['in_csv'])
    safe_asset = asset.replace('/', '_').replace('\\', '_').replace(':', '_').replace('*', '_')
    run_name = f'ml_outputs_rescue_{safe_asset}_{core}_wfcal'
    out_dir = ml_rescue_root / run_name
    out_dir.mkdir(parents=True, exist_ok=True)

    row = {'asset': asset, 'core': core, 'in_csv': str(in_csv), 'ml_run': run_name, 'status': 'ok'}

    try:
        raw = _load_csv_with_date(in_csv)
        needed = {'Open', 'High', 'Low', 'Close'}
        if not needed.issubset(set(raw.columns)):
            raise ValueError(f'missing OHLC columns in {in_csv.name}')

        X = build_features(raw).dropna(axis=1, how='all')
        y = make_labels(raw['Close'], LabelSpec(horizon=5, task='classification', threshold=0.0015))

        m = (~pd.isna(y))
        X = X.loc[m].dropna(how='all')
        y = y.loc[X.index].astype(int)

        if len(X) < 120:
            row['status'] = 'skipped_small_dataset'
            row['n_rows'] = int(len(X))
            retrain_rows.append(row)
            continue

        splits = _walkforward_splits(len(X), n_splits=N_SPLITS, test_window=TEST_WINDOW, purge=PURGE, min_train=60)
        if len(splits) < 3:
            row['status'] = 'skipped_insufficient_folds'
            row['n_rows'] = int(len(X))
            row['n_folds'] = int(len(splits))
            retrain_rows.append(row)
            continue

        fold_rows = []
        for fi, (tr, te) in enumerate(splits, start=1):
            Xtr, Xte = X.iloc[tr], X.iloc[te]
            ytr, yte = y.iloc[tr], y.iloc[te]

            keep_cols = Xtr.columns[Xtr.notna().any(axis=0)]
            Xtr = Xtr.loc[:, keep_cols]
            Xte = Xte.loc[:, keep_cols]

            base = _make_base(core, rs=42 + fi)
            cal_method = 'isotonic' if len(Xtr) >= 220 else 'sigmoid'
            clf = CalibratedClassifierCV(base, method=cal_method, cv=3)
            clf.fit(Xtr, ytr)

            pred = clf.predict(Xte)
            proba = clf.predict_proba(Xte)
            classes = np.asarray(clf.classes_)
            class_to_idx = {int(c): i for i, c in enumerate(classes.tolist())}

            # Build one-vs-rest probabilities for buy/sell diagnostics
            p_buy = proba[:, class_to_idx[1]] if 1 in class_to_idx else np.zeros(len(Xte), dtype=float)
            p_sell = proba[:, class_to_idx[-1]] if -1 in class_to_idx else np.zeros(len(Xte), dtype=float)

            y_enc = pd.Series(yte).replace({-1: 0, 0: 1, 1: 2}).astype(int).to_numpy()
            proba_enc = np.zeros((len(Xte), 3), dtype=float)
            for lab, enc_idx in [(-1, 0), (0, 1), (1, 2)]:
                if lab in class_to_idx:
                    proba_enc[:, enc_idx] = proba[:, class_to_idx[lab]]

            ll = float(log_loss(y_enc, np.clip(proba_enc, 1e-9, 1.0), labels=[0, 1, 2]))
            y_onehot = np.eye(3)[y_enc]
            brier_multi = float(np.mean(np.sum((proba_enc - y_onehot) ** 2, axis=1)))
            ece = _ece_multiclass(y_enc, proba_enc, bins=10)

            fold_rows.append({
                'fold': fi,
                'train_end': str(Xtr.index[-1]),
                'test_start': str(Xte.index[0]),
                'test_end': str(Xte.index[-1]),
                'n_train': int(len(Xtr)),
                'n_test': int(len(Xte)),
                'accuracy': float(accuracy_score(yte, pred)),
                'balanced_accuracy': float(balanced_accuracy_score(yte, pred)),
                'f1_macro': float(f1_score(yte, pred, average='macro')),
                'cal_log_loss': ll,
                'cal_brier_multi': brier_multi,
                'cal_ece': float(ece),
                'cal_buy_prob_mean': float(np.nanmean(p_buy)) if len(p_buy) else float('nan'),
                'cal_buy_freq': float(np.nanmean((np.asarray(yte) == 1).astype(float))) if len(yte) else float('nan'),
                'cal_sell_prob_mean': float(np.nanmean(p_sell)) if len(p_sell) else float('nan'),
                'cal_sell_freq': float(np.nanmean((np.asarray(yte) == -1).astype(float))) if len(yte) else float('nan'),
                'cal_buy_gap': float(abs(np.nanmean(p_buy) - np.nanmean((np.asarray(yte) == 1).astype(float)))) if len(yte) else float('nan'),
                'cal_sell_gap': float(abs(np.nanmean(p_sell) - np.nanmean((np.asarray(yte) == -1).astype(float)))) if len(yte) else float('nan'),
                'cal_method': cal_method,
            })

        cv_df = pd.DataFrame(fold_rows)
        cv_path = out_dir / 'cv_metrics.csv'
        cv_df.to_csv(cv_path, index=False)

        # Final calibrated artifact for linkage/deployment
        keep_all = X.columns[X.notna().any(axis=0)]
        X_fit = X.loc[:, keep_all]
        y_fit = y.astype(int)
        base_final = _make_base(core, rs=123)
        final_method = 'isotonic' if len(X_fit) >= 220 else 'sigmoid'
        final_clf = CalibratedClassifierCV(base_final, method=final_method, cv=3)
        final_clf.fit(X_fit, y_fit)
        model_path = out_dir / 'model.joblib'
        dump(final_clf, model_path)

        metrics = {
            'input': str(in_csv),
            'inputs': None,
            'feature_count': int(X_fit.shape[1]),
            'mode': 'walkforward',
            'task': 'classification',
            'model': core,
            'n_folds': int(len(cv_df)),
            'purge': int(PURGE),
            'test_window': int(TEST_WINDOW),
            'cv_metrics_csv': str(cv_path),
            'calibration_method': final_method,
            'cal_ece': float(pd.to_numeric(cv_df['cal_ece'], errors='coerce').mean()),
            'cal_brier_multi': float(pd.to_numeric(cv_df['cal_brier_multi'], errors='coerce').mean()),
            'cal_log_loss': float(pd.to_numeric(cv_df['cal_log_loss'], errors='coerce').mean()),
        }
        (out_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')

        row.update({
            'n_rows': int(len(X)),
            'n_folds': int(len(cv_df)),
            'f1_macro_mean': float(pd.to_numeric(cv_df['f1_macro'], errors='coerce').mean()),
            'cal_ece_mean': float(pd.to_numeric(cv_df['cal_ece'], errors='coerce').mean()),
            'cal_brier_mean': float(pd.to_numeric(cv_df['cal_brier_multi'], errors='coerce').mean()),
            'cal_logloss_mean': float(pd.to_numeric(cv_df['cal_log_loss'], errors='coerce').mean()),
            'model_path': str(model_path),
        })

    except Exception as e:
        row['status'] = 'error'
        row['error'] = str(e)

    retrain_rows.append(row)

retrain_df = pd.DataFrame(retrain_rows)
retrain_out = OUT_DIR / 'ml_rescue_retraining_report.csv'
retrain_df.to_csv(retrain_out, index=False)

display(group_df)
display(retrain_df)
print('Saved ML rescue retraining report:', retrain_out)

In [ ]:
# Rerun the same rescue loop, link reruns to calibrated ML artifacts, and rescore governance
from concurrent.futures import ThreadPoolExecutor, as_completed
import shlex

TOP_N_RESCUE_RERUN = 10
MAX_PARALLEL_RESCUE_RERUN = 4

uplift_df_rerun = pd.read_csv(OUT_DIR / 'candidate_uplift_rankings.csv').head(TOP_N_RESCUE_RERUN).copy()
candidate_runs_rerun = uplift_df_rerun['run'].tolist()


def _resolve_input_path(raw_input: str) -> Path | None:
    if not raw_input:
        return None
    p = Path(raw_input)
    candidates = [ROOT / p, ROOT / 'artifacts' / 'outputs' / p, ROOT / 'artifacts' / p]
    for c in candidates:
        if c.exists() and c.is_file():
            return c.resolve()
    target_name = p.name
    for c in (ROOT / 'artifacts').rglob(target_name):
        if c.is_file():
            return c.resolve()
    return None


meta_rows_rerun = []
for run in candidate_runs_rerun:
    summary_json = ROOT / 'artifacts' / 'backtests' / run / 'backtest_summary.json'
    if not summary_json.exists():
        continue
    s = json.loads(summary_json.read_text(encoding='utf-8'))
    raw_input = s.get('input')
    variant = s.get('variant')
    asset = s.get('asset')
    in_csv = _resolve_input_path(raw_input)
    if variant and in_csv is not None:
        meta_rows_rerun.append({
            'run': run,
            'asset': str(asset or 'UNKNOWN'),
            'variant': variant,
            'core': variant.split('_', 1)[0] if '_' in variant else str(s.get('model') or '').lower(),
            'input_raw': raw_input,
            'in_csv': str(in_csv),
        })

meta_rerun_df = pd.DataFrame(meta_rows_rerun)
if meta_rerun_df.empty:
    raise RuntimeError('No resolvable candidate metadata found for ML-rescue rerun loop.')

# Build model-link map from retraining outputs
if 'retrain_df' not in globals():
    retrain_df = pd.read_csv(OUT_DIR / 'ml_rescue_retraining_report.csv')
model_link_df = retrain_df[(retrain_df['status'] == 'ok') & retrain_df['model_path'].notna()].copy() if 'model_path' in retrain_df.columns else pd.DataFrame()
model_link_map = {(str(r['asset']), str(r['core']).lower()): str(r['model_path']) for _, r in model_link_df.iterrows()}

by_input = defaultdict(set)
for _, r in meta_rerun_df.iterrows():
    by_input[str(r['in_csv'])].add(str(r['variant']))

strategy_sweep_py = ROOT / 'projects' / 'strategy_sweep.py'
assert strategy_sweep_py.exists(), f'Missing script: {strategy_sweep_py}'

rescue_root = ROOT / 'artifacts' / 'backtests'
rescue_root.mkdir(parents=True, exist_ok=True)


def _run_rescue_group(in_csv: str, variants: list[str]) -> dict:
    cores = sorted({v.split('_', 1)[0] for v in variants if '_' in v})
    cmd = [
        sys.executable, '-m', 'projects.strategy_sweep',
        '--in-csv', in_csv,
        '--out-dir', str(rescue_root),
        '--out-prefix', 'bt_rescue_ml',
        '--cores', ','.join(cores) if cores else 'hgb,rf,mlp',
        '--only-variants', ','.join(sorted(set(variants))),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd=str(ROOT))
    return {
        'in_csv': in_csv,
        'variants': ','.join(sorted(set(variants))),
        'return_code': proc.returncode,
        'stdout_tail': proc.stdout[-3000:],
        'stderr_tail': proc.stderr[-3000:],
        'cmd': ' '.join(shlex.quote(x) for x in cmd),
    }

jobs = [(k, sorted(v)) for k, v in by_input.items()]
results = []
with ThreadPoolExecutor(max_workers=min(MAX_PARALLEL_RESCUE_RERUN, len(jobs))) as ex:
    futs = [ex.submit(_run_rescue_group, in_csv, variants) for in_csv, variants in jobs]
    for fut in as_completed(futs):
        results.append(fut.result())

rescue_ml_exec_df = pd.DataFrame(results).sort_values(['return_code','in_csv']).reset_index(drop=True)
rescue_ml_exec_out = OUT_DIR / 'rescue_ml_execution_report.csv'
rescue_ml_exec_df.to_csv(rescue_ml_exec_out, index=False)

# Link generated bt_rescue_ml summaries to calibrated model artifacts for ML-gate merge
linked = 0
missing_link = 0
for bt_dir in rescue_root.glob('bt_rescue_ml_*'):
    js = bt_dir / 'backtest_summary.json'
    if not js.exists():
        continue
    try:
        s = json.loads(js.read_text(encoding='utf-8'))
        asset = str(s.get('asset') or 'UNKNOWN')
        variant = str(s.get('variant') or '')
        core = variant.split('_', 1)[0] if '_' in variant else str(s.get('model') or '').lower()
        model_path = model_link_map.get((asset, core.lower()))
        if model_path:
            s['model'] = model_path
            s['model_link_mode'] = 'ml_rescue_calibrated'
            js.write_text(json.dumps(s, indent=2), encoding='utf-8')
            linked += 1
        else:
            missing_link += 1
    except Exception:
        missing_link += 1

# Rescore after ML-rescue rerun
rescue_ml_out_dir = ROOT / 'master_brain_rescue_ml'
rescue_ml_out_dir.mkdir(parents=True, exist_ok=True)

rescore_ml_cmd = [
    sys.executable,
    str(PIPELINE),
    '--in-root', str(ROOT),
    '--out-dir', str(rescue_ml_out_dir),
    '--gate-stress-bps', '20',
    '--stress-bps-grid', '10,20,35',
]
rescore_ml = subprocess.run(rescore_ml_cmd, capture_output=True, text=True, cwd=str(ROOT))
print('ml-rescore return_code:', rescore_ml.returncode)
print(rescore_ml.stdout[-3000:])
if rescore_ml.returncode != 0:
    print('stderr_tail:')
    print(rescore_ml.stderr[-3000:])
    raise RuntimeError('Rescore after ML-rescue rerun failed')

baseline_summary = json.loads((OUT_DIR / 'master_brain_summary.json').read_text(encoding='utf-8'))
rescue_summary = json.loads((ROOT / 'master_brain_rescue' / 'master_brain_summary.json').read_text(encoding='utf-8')) if (ROOT / 'master_brain_rescue' / 'master_brain_summary.json').exists() else baseline_summary
ml_rescue_summary = json.loads((rescue_ml_out_dir / 'master_brain_summary.json').read_text(encoding='utf-8'))

b = baseline_summary.get('counts', {})
r = rescue_summary.get('counts', {})
m = ml_rescue_summary.get('counts', {})

cmp3 = pd.DataFrame([
    {'metric': 'backtests_scored', 'baseline': b.get('backtests_scored'), 'rescue': r.get('backtests_scored'), 'ml_rescue': m.get('backtests_scored')},
    {'metric': 'promoted_runs', 'baseline': b.get('promoted_runs'), 'rescue': r.get('promoted_runs'), 'ml_rescue': m.get('promoted_runs')},
    {'metric': 'ml_runs_scored', 'baseline': b.get('ml_runs_scored'), 'rescue': r.get('ml_runs_scored'), 'ml_rescue': m.get('ml_runs_scored')},
])
cmp3['delta_rescue_vs_base'] = pd.to_numeric(cmp3['rescue'], errors='coerce') - pd.to_numeric(cmp3['baseline'], errors='coerce')
cmp3['delta_mlrescue_vs_rescue'] = pd.to_numeric(cmp3['ml_rescue'], errors='coerce') - pd.to_numeric(cmp3['rescue'], errors='coerce')
cmp3['delta_mlrescue_vs_base'] = pd.to_numeric(cmp3['ml_rescue'], errors='coerce') - pd.to_numeric(cmp3['baseline'], errors='coerce')

cmp3_out = rescue_ml_out_dir / 'rescue_ml_before_after_comparison.csv'
cmp3.to_csv(cmp3_out, index=False)

ml_report_lines = [
    '# ML-Rescue Loop Outcome Summary',
    '',
    f'- Retraining runs attempted: **{len(retrain_df)}**',
    f"- Retraining successful: **{int((retrain_df['status'] == 'ok').sum()) if 'status' in retrain_df.columns else 0}**",
    f"- Rescue rerun jobs: **{len(rescue_ml_exec_df)}**",
    f"- Rescue rerun failures: **{int((rescue_ml_exec_df['return_code'] != 0).sum()) if not rescue_ml_exec_df.empty else 0}**",
    f'- Backtest summaries linked to calibrated models: **{linked}**',
    f'- Backtest summaries without model link: **{missing_link}**',
    '',
    '## Counts comparison',
    '',
    f"- Promoted runs (baseline -> rescue -> ml_rescue): **{b.get('promoted_runs', 0)} -> {r.get('promoted_runs', 0)} -> {m.get('promoted_runs', 0)}**",
    f"- ML runs scored (baseline -> rescue -> ml_rescue): **{b.get('ml_runs_scored', 0)} -> {r.get('ml_runs_scored', 0)} -> {m.get('ml_runs_scored', 0)}**",
    '',
    f'- Comparison CSV: `{cmp3_out}`',
    f'- Retraining report CSV: `{retrain_out}`',
    f'- Rescue rerun execution CSV: `{rescue_ml_exec_out}`',
]

ml_report_md = rescue_ml_out_dir / 'ml_rescue_loop_outcome.md'
ml_report_md.write_text('\n'.join(ml_report_lines) + '\n', encoding='utf-8')

display(meta_rerun_df)
display(rescue_ml_exec_df[['in_csv','variants','return_code']])
display(cmp3)
print('Saved rescue rerun report:', rescue_ml_exec_out)
print('Saved comparison:', cmp3_out)
print('Saved summary:', ml_report_md)